# Multi-Modal Brain Age Ensemble (T1 + T2) su IXI Dataset
Questo notebook carica due modelli SFCN pre-addestrati separatamente (uno su scansioni T1 e uno su scansioni T2). 
Per ogni paziente nel Test Set, carica simultaneamente entrambe le immagini, effettua due predizioni indipendenti, e le unisce (Ensemble) calcolando l'età media predetta.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('./SFCN')

In [ ]:
import os
import pandas as pd
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
from datetime import datetime
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

from dp_model.model_files.sfcn import SFCN
import dp_model.utils as dpu

PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Configurazione Percorsi

In [ ]:
# --- PERCORSI MODELLI ---
MODEL_T1_PATH = "/kaggle/input/sfcn-models/best_model_t1.pth" # Sostituisci con il path al tuo miglior modello T1
MODEL_T2_PATH = "/kaggle/input/sfcn-models/best_model_t2.pth" # Sostituisci con il path al tuo miglior modello T2

# --- PERCORSI DATI ---
CSV_PATH = "/kaggle/input/ixi-dataset/ixi_info.csv"

T1_DATA_DIR = "/kaggle/input/ixi-dataset/Prep_T1" # Cartella root che contiene Prep_Guys_T1, ecc.
T2_DATA_DIR = "/kaggle/input/ixi-dataset/Prep_T2" # Cartella root che contiene Prep_Guys_T2, ecc.

T1_CORRECTED_DIR = "/kaggle/input/datasets/collab4444/ixi-t1-corrected-items/corrected_IXI_subjects_T1w"
T2_CORRECTED_DIR = "/kaggle/input/datasets/collab4444/ixi-t1-corrected-items/corrected_IXI_subjects_T2w"

OUTPUT_DIM = 100
BATCH_SIZE = 1 # Durante il test inferiamo uno alla volta per tracciare meglio le età

## 2. Multi-Modal Dataloader (Allinea T1 e T2)

In [ ]:
class IXIMultiModalDataset(Dataset):
    def __init__(self, t1_dir, t2_dir, csv_path):
        self.samples = []
        self.bin_range = [0, OUTPUT_DIM]
        self.bin_step = 1
        self.sigma = 1.0
        
        if not os.path.exists(csv_path):
            print(f"ATTENZIONE: CSV non trovato in {csv_path}")
            return
            
        df = pd.read_csv(csv_path)
        reference_date = datetime(2015, 2, 23)
        
        # Mappatura cartelle siti IXI
        sites = [('Prep_Guys', 'Guys'), ('Prep_HH', 'HH'), ('Prep_IOP', 'IOP')]
        
        for sf_prefix, sf_corrected in sites:
            # Per T1
            folder_t1 = os.path.join(t1_dir, f"{sf_prefix}_T1")
            if not os.path.exists(folder_t1): 
                folder_t1 = os.path.join(t1_dir, sf_prefix) # Fallback se non c'è il suffisso _T1
                
            # Per T2
            folder_t2 = os.path.join(t2_dir, f"{sf_prefix}_T2")
            
            if not os.path.exists(folder_t1) or not os.path.exists(folder_t2):
                continue
                
            # Iteriamo sui file presenti nella cartella T1 e cerchiamo i corrispondenti in T2
            for file_t1 in os.listdir(folder_t1):
                if not (file_t1.startswith("registered_image_") and file_t1.endswith(".nii")):
                    continue
                    
                file_t2 = file_t1 # Il nome file è di solito identico tra le modaltà
                
                nii_path_t1 = os.path.join(folder_t1, file_t1)
                nii_path_t2 = os.path.join(folder_t2, file_t2)
                
                # SOSTITUZIONE FILE CORROTTI (T1)
                corrected_path_t1 = os.path.join(T1_CORRECTED_DIR, sf_corrected, file_t1)
                if os.path.exists(corrected_path_t1):
                    nii_path_t1 = corrected_path_t1
                    
                # SOSTITUZIONE FILE CORROTTI (T2)
                corrected_path_t2 = os.path.join(T2_CORRECTED_DIR, sf_corrected, file_t2)
                if os.path.exists(corrected_path_t2):
                    nii_path_t2 = corrected_path_t2
                    
                # Assicuriamoci che ENTRAMBI i file esistano fisicamente
                if not os.path.exists(nii_path_t1) or not os.path.exists(nii_path_t2):
                    continue
                    
                ixi_id_str = file_t1.replace("registered_image_", "").replace(".nii", "")
                try:
                    ixi_id = int(ixi_id_str)
                except ValueError:
                    continue
                    
                row = df[df['IXI_ID'] == ixi_id]
                if len(row) == 0:
                    continue
                    
                dob_str = row.iloc[0]['DOB']
                if pd.isna(dob_str):
                    continue
                    
                try:
                    dob_str = str(dob_str).split(" ")[0]
                    dob = datetime.strptime(dob_str, "%Y-%m-%d")
                    true_age = (reference_date - dob).days / 365.25
                    
                    y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
                    
                    self.samples.append({
                        "ixi_id": ixi_id,
                        "nii_path_t1": nii_path_t1,
                        "nii_path_t2": nii_path_t2,
                        "label_vect": y,
                        "true_age": true_age
                    })
                except Exception as e:
                    continue
                    
        print(f"Trovati {len(self.samples)} pazienti con doppia modalità T1 e T2.")

    def __len__(self):
        return len(self.samples)
        
    def _process_image(self, nii_path):
        img = nib.load(nii_path)
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        x_c = int((in_sp[0] - out_sp[0]) / 2)
        y_c = int((in_sp[1] - out_sp[1]) / 2)
        z_c = int((in_sp[2] - out_sp[2]) / 2)
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        data = np.expand_dims(data, axis=0)
        
        return torch.from_numpy(data)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        tensor_t1 = self._process_image(sample['nii_path_t1'])
        tensor_t2 = self._process_image(sample['nii_path_t2'])
        
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_t1, tensor_t2, label_vect, sample['true_age'], sample['ixi_id']

## 3. Selezione Test Set
Siccome i modelli sono stati addestrati su uno split 60-20-20, dobbiamo replicare quello split (usando lo stesso `random_state`) per isolare i pazienti che la rete **non ha mai visto**.

In [ ]:
dataset = IXIMultiModalDataset(T1_DATA_DIR, T2_DATA_DIR, CSV_PATH)
dataset_size = len(dataset)

if dataset_size > 0:
    # Riproduciamo lo split originale per ottenere l'esatto Test Set (20%)
    all_indices = np.arange(dataset_size)
    
    train_pool_idx, test_idx = train_test_split(
        all_indices, 
        test_size=0.20, random_state=42
    )
    
    # Ulteriore split originale per isolare Test da Val
    train_idx, val_idx = train_test_split(
        train_pool_idx, 
        test_size=0.25, random_state=42 
    )
    
    test_dataset = torch.utils.data.Subset(dataset, test_idx)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    print(f"Pazienti Isolati nel Test Set: {len(test_dataset)}")
else:
    print("Impossibile continuare: Dati non trovati.")

## 4. Inizializzazione Modelli

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model(path):
    model = SFCN(output_dim=OUTPUT_DIM)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.load_state_dict(torch.load(path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

if dataset_size > 0:
    print("Caricamento Modello T1...")
    model_t1 = load_model(MODEL_T1_PATH)
    
    print("Caricamento Modello T2...")
    model_t2 = load_model(MODEL_T2_PATH)
    
    print("Modelli caricati correttamente e pronti per l'inferenza.")

## 5. Esecuzione Multi-Modal Ensemble

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print(" INIZIO INFERENZA TEST SET E CONFRONTO")
    print("==============================================")
    
    bin_centers = np.arange(0, OUTPUT_DIM, 1)
    
    true_ages = []
    preds_t1 = []
    preds_t2 = []
    preds_ensemble = []
    
    with torch.no_grad():
        for tensor_t1, tensor_t2, label_vect, true_age, ixi_id in test_loader:
            tensor_t1 = tensor_t1.to(device)
            tensor_t2 = tensor_t2.to(device)
            
            # Predizione T1
            out_t1 = model_t1(tensor_t1)[0].reshape(tensor_t1.size(0), -1)
            prob_t1 = torch.exp(out_t1).cpu().numpy()
            pred_age_t1 = np.dot(prob_t1, bin_centers)[0]
            
            # Predizione T2
            out_t2 = model_t2(tensor_t2)[0].reshape(tensor_t2.size(0), -1)
            prob_t2 = torch.exp(out_t2).cpu().numpy()
            pred_age_t2 = np.dot(prob_t2, bin_centers)[0]
            
            # Ensemble (Media Predizioni)
            pred_age_ens = (pred_age_t1 + pred_age_t2) / 2.0
            
            true_ages.append(true_age.item())
            preds_t1.append(pred_age_t1)
            preds_t2.append(pred_age_t2)
            preds_ensemble.append(pred_age_ens)
            
            print(f"IXI_{ixi_id.item():03d} | Reale: {true_age.item():.1f} | T1: {pred_age_t1:.1f} | T2: {pred_age_t2:.1f} | Ens: {pred_age_ens:.1f}")
            
    # --- CALCOLO METRICHE ---
    true_ages = np.array(true_ages)
    preds_t1 = np.array(preds_t1)
    preds_t2 = np.array(preds_t2)
    preds_ensemble = np.array(preds_ensemble)
    
    def calc_metrics(y_true, y_pred):
        mae = np.mean(np.abs(y_true - y_pred))
        corr, _ = pearsonr(y_true, y_pred)
        return mae, corr
        
    mae_t1, corr_t1 = calc_metrics(true_ages, preds_t1)
    mae_t2, corr_t2 = calc_metrics(true_ages, preds_t2)
    mae_ens, corr_ens = calc_metrics(true_ages, preds_ensemble)
    
    print("\n>>> RISULTATI FINALI (TEST SET) <<<")
    print(f"Modello T1  -> MAE: {mae_t1:.3f} anni | Pearson r: {corr_t1:.3f}")
    print(f"Modello T2  -> MAE: {mae_t2:.3f} anni | Pearson r: {corr_t2:.3f}")
    print(f"ENSEMBLE    -> MAE: {mae_ens:.3f} anni | Pearson r: {corr_ens:.3f}")
    
    # --- PLOT RISULTATI ---
    plt.figure(figsize=(18, 5))
    
    # T1
    plt.subplot(1, 3, 1)
    plt.scatter(true_ages, preds_t1, color='blue', alpha=0.6)
    plt.plot([min(true_ages), max(true_ages)], [min(true_ages), max(true_ages)], 'r--')
    plt.title(f"T1 Solo (MAE: {mae_t1:.2f})")
    plt.xlabel("Età Reale")
    plt.ylabel("Età Predetta")
    
    # T2
    plt.subplot(1, 3, 2)
    plt.scatter(true_ages, preds_t2, color='green', alpha=0.6)
    plt.plot([min(true_ages), max(true_ages)], [min(true_ages), max(true_ages)], 'r--')
    plt.title(f"T2 Solo (MAE: {mae_t2:.2f})")
    plt.xlabel("Età Reale")
    
    # Ensemble
    plt.subplot(1, 3, 3)
    plt.scatter(true_ages, preds_ensemble, color='purple', alpha=0.6)
    plt.plot([min(true_ages), max(true_ages)], [min(true_ages), max(true_ages)], 'r--')
    plt.title(f"Ensemble T1+T2 (MAE: {mae_ens:.2f})")
    plt.xlabel("Età Reale")
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'multimodal_ensemble_results.png'), dpi=300)
    plt.show()